In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import IPython.display as ipd

sns.set_theme(style="white")

In [ ]:
from pathlib import Path

data_dir = next(
    (path for path in (Path("data"), Path("../data")) if (path / "train.csv").is_file()),
    None,
)

audio_dir = data_dir / "DATASET"
audio_files = sorted(audio_dir.glob("*.wav"))
len(audio_files)

In [ ]:
data = pd.read_csv(data_dir / "train.csv")
data["audio_filename"] = data["filename"].str.replace(
    "sentence_kaa0", "sentence_kaa1", n=1, regex=False
)
available_audio = {path.name for path in audio_files}
data = data.loc[data["audio_filename"].isin(available_audio)].copy().reset_index(drop=True)


len(data)

In [ ]:
import re
import subprocess
import wave


def get_audio_info(audio_filename):
    recording_path = audio_dir / audio_filename
    with recording_path.open("rb") as audio_file:
        header = audio_file.read(12)

    if header[:4] == b"RIFF" and header[8:12] == b"WAVE":
        with wave.open(str(recording_path), "rb") as recording:
            sample_rate = recording.getframerate()
            return recording.getnframes() / sample_rate, sample_rate

    if header[4:8] == b"ftyp":
        result = subprocess.run(
            ["afinfo", str(recording_path)], capture_output=True, text=True, check=True
        )
        duration = re.search(r"estimated duration:\s*([\d.]+)\s*sec", result.stdout)
        sample_rate = re.search(r"(\d+)\s+Hz", result.stdout)
        if duration and sample_rate:
            return float(duration.group(1)), int(sample_rate.group(1))
        raise ValueError(f"Could not read audio metadata from {recording_path}")

    raise ValueError(f"Unsupported audio format: {recording_path}")


data[["duration", "original_sampling_rate"]] = pd.DataFrame(
    data["audio_filename"].map(get_audio_info).tolist(), index=data.index
)
data["original_sampling_rate"] = data["original_sampling_rate"].astype(int)

In [ ]:
data.head(10)

In [ ]:
data.describe()

In [ ]:
data["duration"].describe()

In [ ]:
data.head(20)

In [ ]:
number = 6

text = data.iloc[(number-1),1]

print(f"number: {number}")
print(f"\ntext: {text}")

In [ ]:
texts = data.loc[:, "text"].str.startswith("aqsana", na=False)
texts.value_counts()

In [ ]:
total_duration_seconds = data["duration"].sum()
print(f"Total duration: {total_duration_seconds:.1f} seconds ({total_duration_seconds / 3600:.2f} hours)")

In [ ]:
data["duration"].sum()

In [ ]:
ipd.Audio(audio_files[0])

In [ ]:
y, sr = librosa.load(audio_files[9], sr=None)

sr

In [ ]:
pd.Series(y).plot(figsize=(10,5), lw=1)

plt.show()

In [ ]:
audio_path = audio_files[0]

audio, sample_rate = librosa.load(
    audio_path,
    sr=None,
    mono=True
)

print("File:", audio_path.name)
print("Sample rate:", sample_rate)
print("Number of samples:", len(audio))
print("Duration:", len(audio) / sample_rate)

In [ ]:
target_sample_rate = 16000
resampled_audio = librosa.resample(
    audio,
    orig_sr=sample_rate,
    target_sr=target_sample_rate,
)
print(f"Resampled example: {sample_rate} Hz -> {target_sample_rate} Hz")
print(f"Samples: {len(audio)} -> {len(resampled_audio)}")

In [ ]:
data["original_sampling_rate"].value_counts().sort_index()